In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")

    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("rafi")
    driver.find_element(By.ID, "password").send_keys("787878")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)

    # Open Dashboard through the real sidebar (label verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'Dashboard')]"))).click()
    time.sleep(2)

    # Real dashboard markers (verified in DashboardContainer.jsx, widgets.jsx, StaffApp.jsx)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//section[@aria-label='Admin dashboard']")))
    body = driver.find_element(By.TAG_NAME, "body").text
    assert "Overview of your pharmacy operations today" in body
    assert any(g in body for g in ["Good morning", "Good afternoon", "Good evening"])
    assert len(body.strip()) > 200, "Dashboard looks blank."
    assert not driver.find_elements(By.ID, "username"), "Logged out back to the login form."
    print("Dashboard content:", body[:200])
    print("PASS: Dashboard navigation works")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("41_dashboard_navigation_FAIL.png")
finally:
    driver.quit()